# MCP Server Deployment in Bedrock AgentCore Runtime — FSI Edition

This lab demonstrates how to deploy a transaction validation MCP server to Amazon Bedrock AgentCore Runtime.

## Overview

In this lab, you will:
- Create a custom MCP server with FSI tools (transaction validation, sanctions check, customer risk)
- Test the MCP server locally
- Set up authentication using Amazon Cognito
- Deploy the MCP server to Bedrock AgentCore Runtime
- Test the deployed server with Strands Agents

## Why MCP for FSI?

- **Centralized tools** — Deploy once, use from any agent
- **Secure access** — Cognito authentication ensures only authorized agents call your tools
- **Scalable** — Auto-scales with demand (peak trading hours)
- **Auditable** — Every tool call is logged

## Prerequisites

In [ ]:
import os
#os.environ['AWS_ACCESS_KEY_ID'] = ''
#os.environ['AWS_SECRET_ACCESS_KEY'] = ''
#os.environ['AWS_SESSION_TOKEN'] = ''
#os.environ['AWS_REGION'] = ''

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore mcp fastmcp rich

In [17]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'):
    NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'):
    NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'

print(f'Region: {region}')
print(f'Nova Pro Model ID: {NOVA_PRO_MODEL_ID}')

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## The MCP Server: Transaction Validation

Our MCP server (`mcp_server.py`) exposes three FSI tools:

| Tool | Purpose |
|------|--------|
| `validate_transaction` | Check transaction against risk rules (amount, category, customer risk) |
| `check_sanctions` | Verify entity against sanctions lists (OFAC, UN, AU DFAT) |
| `get_customer_risk_profile` | Retrieve customer risk level, KYC status, daily limits |

Let's look at the server code:

In [18]:
# View the MCP server code
with open('mcp_server.py', 'r') as f:
    print(f.read())

from mcp.server.fastmcp import FastMCP
import json
from datetime import datetime

mcp = FastMCP(host="0.0.0.0", stateless_http=True)


@mcp.tool()
def validate_transaction(amount: float, currency: str = "AUD", merchant_category: str = "general", customer_risk_level: str = "medium") -> dict:
    """Validate a financial transaction against risk rules.
    Args:
        amount (float): Transaction amount.
        currency (str): Currency code (AUD, USD, EUR, GBP).
        merchant_category (str): Category: general, gambling, crypto, high_risk, electronics.
        customer_risk_level (str): Customer risk: low, medium, high.
    Returns:
        Dictionary with validation result, risk score, and any flags.
    """
    flags = []
    risk_score = 0

    # Amount-based rules
    if amount > 10000:
        flags.append("HIGH_VALUE_TRANSACTION")
        risk_score += 3
    if amount > 50000:
        flags.append("REPORTING_THRESHOLD_EXCEEDED")
        risk_score += 2

    # Category-based rule

## Local Testing (Optional)

Before deploying, you can test locally:

**In a separate terminal:**
```bash
cd 04-agentcore-runtime-mcp-fsi/
pip install mcp fastmcp
python mcp_server.py
```

Then run the cell below to test:

In [ ]:
# Test locally (only if MCP server is running on localhost:8000)
# Skip this cell if port 8000 is not available

from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

try:
    mcp_url = 'http://localhost:8000/mcp'
    mcp_client = MCPClient(lambda: streamablehttp_client(mcp_url))

    with mcp_client:
        tools = mcp_client.list_tools_sync()
        print(f'Available tools: {[t.tool_name for t in tools]}')

        agent = Agent(
            model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
            system_prompt='You are a transaction risk analyst. Use the available tools to validate transactions.',
            tools=tools,
        )
        agent('Validate a $15,000 AUD transaction to a crypto exchange for customer CUST-4421')
except Exception as e:
    print(f'Local server not running (expected if skipping): {e}')

## Deploy to Bedrock AgentCore Runtime

Now we'll deploy the MCP server to AgentCore Runtime with Cognito authentication.

### Step 1: Set up Amazon Cognito for Authentication

In [19]:
import boto3
import json

cognito_client = boto3.client('cognito-idp', region_name=region)

# Create User Pool
pool_name = 'fsi_mcp_auth_pool'
try:
    pool_response = cognito_client.create_user_pool(
        PoolName=pool_name,
        Policies={'PasswordPolicy': {'MinimumLength': 8}},
        AutoVerifiedAttributes=['email'],
    )
    user_pool_id = pool_response['UserPool']['Id']
    print(f'✅ User Pool created: {user_pool_id}')
except Exception as e:
    if 'already exists' in str(e).lower():
        pools = cognito_client.list_user_pools(MaxResults=50)['UserPools']
        user_pool_id = next(p['Id'] for p in pools if p['Name'] == pool_name)
        print(f'✅ Using existing pool: {user_pool_id}')
    else:
        raise e

# Create App Client
client_response = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName='fsi_mcp_client',
    ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH'],
    GenerateSecret=False,
)
client_id = client_response['UserPoolClient']['ClientId']
print(f'✅ App Client created: {client_id}')

# Create test user
test_password = 'FsiDemo2026!'
try:
    cognito_client.admin_create_user(
        UserPoolId=user_pool_id,
        Username='fsi_agent',
        TemporaryPassword=test_password,
        MessageAction='SUPPRESS',
    )
    cognito_client.admin_set_user_password(
        UserPoolId=user_pool_id,
        Username='fsi_agent',
        Password=test_password,
        Permanent=True,
    )
    print(f'✅ Test user created: fsi_agent')
except cognito_client.exceptions.UsernameExistsException:
    print(f'✅ Test user already exists: fsi_agent')

✅ User Pool created: ap-southeast-2_hmJ1Vwc4Q
✅ App Client created: 5cnk49sm9cgqebc33nocu50i95
✅ Test user created: fsi_agent


### Step 2: Configure AgentCore Runtime Deployment

Generate the deployment artifacts (Dockerfile, config):

In [20]:
# The starter toolkit automatically generates Dockerfile and config
# during the configure() step below. No manual setup needed.
print('✅ Step 2 handled automatically by the toolkit in Step 3')

✅ Step 2 handled automatically by the toolkit in Step 3


### Step 3: Deploy to AgentCore Runtime

Launch the deployment using the AgentCore CLI/SDK:

In [21]:
from bedrock_agentcore_starter_toolkit import Runtime
import boto3

region = boto3.session.Session().region_name
print(f'Using AWS region: {region}')

agentcore_runtime = Runtime()

print('Configuring AgentCore Runtime...')
response = agentcore_runtime.configure(
    entrypoint='mcp_server.py',
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file='requirements.txt',
    region=region,
    protocol='MCP',
    agent_name='fsi_transaction_validator',
    non_interactive=True,
    authorizer_configuration={
        'customJWTAuthorizer': {
            'allowedClients': [client_id],
            'discoveryUrl': f'https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration',
        }
    }
)
print('Configuration completed ✓')

print('\n🚀 Launching deployment (3-5 minutes)...')
launch_result = agentcore_runtime.launch()
mcp_runtime_arn = launch_result.agent_arn
print(f'✅ Launched! Agent ARN: {mcp_runtime_arn}')

Entrypoint parsed: file=/Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/04-agentcore-runtime-mcp-fsi/mcp_server.py, bedrock_agentcore_name=mcp_server
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: fsi_transaction_validator


Using AWS region: ap-southeast-2
Configuring AgentCore Runtime...


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: /Users/zohaibso/AI 
Workshops/FSI-AgentCore-Workshop/04-agentcore-runtime-mcp-fsi/Dockerfile

Keeping 'fsi_transaction_validator' as default agent
Bedrock AgentCore configured: /Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/04-agentcore-runtime-mcp-fsi/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'fsi_transaction_validator' to account 905035168378 (ap-southeast-2)
Generated image tag: 20260601-035135-939
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: fsi_transaction_validator
ECR repository available: 905035168378.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-fsi_transaction_validator
Getting or creating execution role for agent

Configuration completed ✓

🚀 Launching deployment (3-5 minutes)...
✅ Reusing existing ECR repository: 905035168378.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-fsi_transaction_validator


✅ Reusing existing execution role: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-2-99ebe9746e
Execution role available: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-2-99ebe9746e
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: fsi_transaction_validator
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-2-99ebe9746e
Reusing existing CodeBuild execution role: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-2-99ebe9746e
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: fsi_transaction_validator/source.zip
Updated CodeBuild project: bedrock-agentcore-fsi_transaction_validator-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 6.3s

✅ Launched! Agent ARN: arn:aws:bedrock-agentcore:ap-southeast-2:905035168378:runtime/fsi_transaction_validator-QJFUt9HetD


### Step 4: Test the Deployed MCP Server

Connect to the deployed server with Cognito authentication:

In [22]:
# Get Cognito access token
auth_response = cognito_client.initiate_auth(
    ClientId=client_id,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={
        'USERNAME': 'fsi_agent',
        'PASSWORD': test_password,
    },
)
access_token = auth_response['AuthenticationResult']['AccessToken']
print(f'✅ Authenticated. Token: {access_token[:20]}...')

✅ Authenticated. Token: eyJraWQiOiJWZUU1a01E...


In [23]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

region = boto3.session.Session().region_name
encoded_arn = mcp_runtime_arn.replace(':', '%3A').replace('/', '%2F')

# Connect to deployed MCP server
mcp_url = f'https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations'
headers = {'Authorization': f'Bearer {access_token}'}

mcp_client = MCPClient(lambda: streamablehttp_client(mcp_url, headers=headers))

with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f'Available tools: {[t.tool_name for t in tools]}')

    agent = Agent(
        model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
        system_prompt='You are a transaction risk analyst. Validate transactions and check sanctions. Be concise.',
        tools=tools,
    )

    agent('Validate a $25,000 AUD transaction to a crypto exchange for customer CUST-4421. Also check if "Shadow Finance Ltd" is sanctioned.')

Available tools: ['validate_transaction', 'check_sanctions', 'get_customer_risk_profile']
<thinking> To address the user's request, I need to validate the transaction and check if the entity is sanctioned. First, I will retrieve the customer risk profile for customer CUST-4421. Then, I will use the retrieved risk level to validate the transaction. Finally, I will check if "Shadow Finance Ltd" is sanctioned. </thinking>

Tool #1: get_customer_risk_profile
<thinking> The customer risk level for CUST-4421 is high. I will now validate the $25,000 AUD transaction to a crypto exchange using this risk level. Additionally, I will check if "Shadow Finance Ltd" is sanctioned. </thinking> 
Tool #2: validate_transaction

Tool #3: check_sanctions
The $25,000 AUD transaction to a crypto exchange for customer CUST-4421 has been **blocked** due to high risk. The transaction triggered the following flags: HIGH_VALUE_TRANSACTION and HIGH_RISK_CATEGORY_CRYPTO.

Additionally, "Shadow Finance Ltd" is **san

Session termination failed: 404


, and EU Sanctions lists. Any transaction involving this entity should be blocked.

## Examining the Agent Loop

The agent loop is how Strands Agents process requests:

1. **Receive** user input
2. **Reason** using the LLM (decide what to do)
3. **Act** by calling a tool
4. **Observe** the tool result
5. **Repeat** or respond to the user

The table below shows each message in the loop — what the model said, which tool it called, and what result it received:

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print(f'Agent Loop Cycles: {agent.event_loop_metrics.cycle_count}' if hasattr(agent, 'event_loop_metrics') else '')

# Get the last agent used in this notebook
active_agent = [v for v in dir() if not v.startswith('_')]

table = Table(title='Agent Messages', show_lines=True)
table.add_column('Role', style='green', width=10)
table.add_column('Text', style='magenta', max_width=50)
table.add_column('Tool', style='cyan', width=20)
table.add_column('Input', style='cyan', max_width=30)
table.add_column('Result', style='cyan', max_width=30)

for msg in agent.messages[-6:]:  # Show last 6 messages for readability
    text = [c['text'] for c in msg['content'] if 'text' in c]
    tool_name = [c['toolUse']['name'] for c in msg['content'] if 'toolUse' in c]
    tool_input = [c['toolUse']['input'] for c in msg['content'] if 'toolUse' in c]
    tool_result = [c['toolResult']['content'][0] for c in msg['content'] if 'toolResult' in c]
    table.add_row(
        msg['role'],
        (text[-1][:100] + '...') if text and len(text[-1]) > 100 else (text[-1] if text else ''),
        tool_name[-1] if tool_name else '',
        (json.dumps(tool_input[-1])[:80] + '...') if tool_input else '',
        (json.dumps(tool_result[-1])[:80] + '...') if tool_result else '',
    )

console.print(table)


## Resource Cleanup (Optional)

In [ ]:
# Uncomment to clean up
# agentcore_runtime.delete()
# cognito_client.delete_user_pool(UserPoolId=user_pool_id)
# print('✅ Resources cleaned up')

## Summary

In this lab, you:

- ✅ Created an FSI MCP server (transaction validation, sanctions, customer risk)
- ✅ Set up Cognito authentication for secure access
- ✅ Deployed to AgentCore Runtime as a managed service
- ✅ Connected a Strands Agent to the deployed server with auth

### FSI Takeaways

| Capability | FSI Value |
|-----------|----------|
| Managed deployment | No infrastructure to maintain |
| Cognito auth | Only authorized agents can validate transactions |
| Auto-scaling | Handles peak trading volumes |
| Centralized tools | One deployment serves all agents in the org |

### Next: Lab 05 — Observability
We'll add full tracing and audit trails to the deployed agent for compliance.